In [11]:
# ============================================================
# SAR Pipeline v2 — Full Analysis with GroupKFold + Validation
# Changes from v1:
#   - Section 4: GroupKFold(5) by farmer ID (Option Y)
#   - Section 3e: k-sensitivity analysis (k=2..5)
#   - Section 3f: GMM clustering robustness check
#   - Section 6b: Surrogate tree 80/20 held-out validation
#   - Section 6c: Surrogate tree 5-fold CV
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
import re
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import (
    cross_val_score, GroupKFold, StratifiedKFold,
    GroupShuffleSplit, cross_validate
)
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    accuracy_score, classification_report
)
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_graphviz
from xgboost import XGBRegressor
from scipy.stats import spearmanr

import os
import subprocess
from pyprojroot import here
OUT = 'figures'
os.makedirs(OUT, exist_ok=True)


# ── PeerJ / journal figure defaults ──────────────────────────────────────
# PNG files are rendered at 600 dpi; all figures are also saved as vector PDF.
JOURNAL_DPI = 600
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.titleweight': 'bold',
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.titlesize': 18,
    'figure.titleweight': 'bold',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.dpi': JOURNAL_DPI,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
})

def save_journal_figure(fig, stem, dpi=JOURNAL_DPI):
    """Save publication-grade raster and vector versions."""
    png_path = os.path.join(OUT, f'{stem}.png')
    pdf_path = os.path.join(OUT, f'{stem}.pdf')
    fig.savefig(png_path, dpi=dpi, bbox_inches='tight',
                facecolor='white', pad_inches=0.08)
    fig.savefig(pdf_path, bbox_inches='tight',
                facecolor='white', pad_inches=0.08)
    print(f'✓ {png_path} and {pdf_path}')

print("✓ Imports complete")

✓ Imports complete


In [12]:
# ============================================================
# SECTION 1 — DATA LOADING & FEATURE DEFINITIONS
# ============================================================
# หาตำแหน่งของโฟลเดอร์ที่ไฟล์ code นี้วางอยู่
base_path = here()
file_path = os.path.join(base_path, "datas", "SFProgramDataPanal.csv")

df = pd.read_csv(file_path)
df = df.replace('.', np.nan)

CLUSTER_FEATS = [
    'age','edu','agri_long','irriga','loan',
    'Avg_ProdManage','Avg_InputManage','Avg_Tech',
    'Avg_Ana&Plan','Avg_Mkting','Avg_Network',
    'Ave_ProdRisk','Ave_InputRisk','Ave_MktRisk','Ave_FinRisk'
]
RF_FEATS = [
    'age','edu','agri_long','gender','irriga','loan','region',
    'Avg_ProdManage','Avg_InputManage','Avg_Tech','Avg_Ana&Plan',
    'Avg_Mkting','Avg_Network',
    'Ave_ProdRisk','Ave_InputRisk','Ave_MktRisk','Ave_FinRisk',
    'sf_participant'
]
SHAP_FEATS = [
    'age','edu','agri_long','irriga','loan',
    'Avg_ProdManage','Avg_Tech','Avg_Mkting',
    'Ave_MktRisk','Ave_FinRisk','Overview_Risk',
    'agri Org_mem','gov_support','All_Skill'
]
SHAP_RENAME = {
    'age':'Age','edu':'Education','agri_long':'Agri Exp',
    'irriga':'Irrigation','loan':'Loan',
    'Avg_ProdManage':'Avg_ProdManage','Avg_Tech':'Avg_Tech',
    'Avg_Mkting':'Avg_Mkting','Ave_MktRisk':'Mkt Risk',
    'Ave_FinRisk':'Fin Risk','Overview_Risk':'Overall Risk',
    'agri Org_mem':'Agri Org','gov_support':'Gov Support',
    'All_Skill':'All_Skill'
}
OUTCOME       = 'Ch_Skill'
FARMER_ID     = 'id'
CLUSTER_NAMES = {0:'Low-skill', 1:'Moderate-skill', 2:'High-skill'}
CLUSTER_COL   = {0:'#4472C4', 1:'#FF8C00', 2:'#2ECC71'}
RF_PARAMS     = dict(n_estimators=500, min_samples_leaf=10,
                     max_features='sqrt', random_state=42, n_jobs=-1)

print(f"Dataset: {df.shape[0]} rows, {df[FARMER_ID].nunique()} unique farmers")
print(f"Panel structure: Year=0 (pre-training), Year=1 (post-training)")

Dataset: 834 rows, 417 unique farmers
Panel structure: Year=0 (pre-training), Year=1 (post-training)


In [13]:
# ============================================================
# SECTION 2 — FARMER SEGMENTATION (K-MEANS, k=3)
# ============================================================

df_cl = df[CLUSTER_FEATS].apply(pd.to_numeric, errors='coerce').dropna().copy()
scaler = StandardScaler()
X_sc   = scaler.fit_transform(df_cl)

# ── 2a. Cluster validity indices k=2..8 ──────────────────────────────────
ks = range(2, 9)
cv_rows = []
for k in ks:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_sc)
    cv_rows.append({
        'k': k,
        'inertia':    km.inertia_,
        'silhouette': silhouette_score(X_sc, lbl),
        'calinski':   calinski_harabasz_score(X_sc, lbl),
        'davies':     davies_bouldin_score(X_sc, lbl),
    })
cv_df = pd.DataFrame(cv_rows)
print("\nCluster validity indices:")
print(cv_df.to_string(index=False, float_format='%.4f'))

# ── 2b. Fit k=3, label clusters ──────────────────────────────────────────
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
df_cl['Cluster_raw'] = km3.fit_predict(X_sc)

df_full = df.loc[df_cl.index].apply(pd.to_numeric, errors='coerce').copy()
df_full['Cluster_raw'] = df_cl['Cluster_raw'].values

order     = df_full.groupby('Cluster_raw')['All_Skill'].mean().sort_values()
label_map = {old: new for new, old in enumerate(order.index)}
df_full['Cluster']      = df_full['Cluster_raw'].map(label_map)
df_full['Cluster_name'] = df_full['Cluster'].map(CLUSTER_NAMES)

print("\nCluster sizes:")
print(df_full['Cluster_name'].value_counts().sort_index())

# ── 2c. Figure: 4-panel validity (publication-ready for PeerJ) ────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
axes = axes.ravel()

# กำหนดขนาดฟอนต์หลักให้ใหญ่ขึ้น
plt.rcParams.update({
    'font.size': 14,          # ขนาดฟอนต์ทั่วไป
    'axes.titlesize': 18,     # ชื่อกราฟแต่ละ panel
    'axes.labelsize': 16,     # label แกน x,y
    'xtick.labelsize': 14,    # tick x
    'ytick.labelsize': 14,    # tick y
    'legend.fontsize': 14,    # legend
})

panels = [
    ('inertia',    'Within-cluster Inertia', 'Elbow Method',       '#1565C0'),
    ('silhouette', 'Silhouette Score',        'Silhouette',         '#1976D2'),
    ('calinski',   'Calinski-Harabasz Score', 'Calinski-Harabasz', '#0277BD'),
    ('davies',     'Davies-Bouldin Score',    'Davies-Bouldin',    '#00796B'),
]

for ax, (col, ylabel, title, color) in zip(axes, panels):
    ax.plot(cv_df['k'], cv_df[col], 'o-', color=color, lw=2.8, ms=8)
    ax.axvline(3, color='#E53935', ls='--', lw=2.2, alpha=0.85, label='k = 3')

    for k, v in zip(cv_df['k'], cv_df[col]):
        ax.annotate(
            f'{v:.3f}', (k, v),
            textcoords='offset points',
            xytext=(0, 10),
            ha='center',
            fontsize=13, fontweight='semibold'
        )

    ax.set_title(title, pad=14, fontweight='bold')
    ax.set_xlabel('Number of clusters (k)')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.legend(frameon=True)

fig.suptitle(
    'Cluster Validity: Four Complementary Indices (k = 2–8)',
    fontsize=22, fontweight='bold', y=1.02
)

plt.tight_layout()
save_journal_figure(fig, 'fig2_elbow_silhouette')
plt.close()

# ── 2d. PCA scatter ───────────────────────────────────────────────────────
pca    = PCA(n_components=2, random_state=42)
X_pca  = pca.fit_transform(X_sc)
ev     = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(8, 7))
for raw_lbl, mapped_lbl in label_map.items():
    mask = df_cl['Cluster_raw'] == raw_lbl
    ax.scatter(X_pca[mask,0], X_pca[mask,1],
               c=CLUSTER_COL[mapped_lbl],
               label=CLUSTER_NAMES[mapped_lbl],
               alpha=0.65, s=25, edgecolors='none')
ax.set_title('Farmer Clusters (k = 3, Silhouette = 0.185)', fontsize=17, pad=12)
ax.set_xlabel(f'PC1 ({ev[0]:.1%})', fontsize=14)
ax.set_ylabel(f'PC2 ({ev[1]:.1%})', fontsize=14)
ax.tick_params(axis='both', labelsize=12)
ax.legend(framealpha=0.9, fontsize=12); ax.grid(alpha=0.25)
plt.tight_layout()
save_journal_figure(fig, 'fig3_clusters_new')
plt.close()


Cluster validity indices:
 k   inertia  silhouette  calinski  davies
 2 9678.1594      0.1986  241.8636  1.8069
 3 8395.1843      0.1845  202.6666  1.8348
 4 7786.4915      0.1712  167.0994  1.7605
 5 7374.7595      0.1581  143.7186  1.9096
 6 6981.7123      0.1433  130.6123  2.0614
 7 6704.5583      0.1448  118.8968  2.0272
 8 6438.7959      0.1393  110.8541  1.9449

Cluster sizes:
Cluster_name
High-skill        334
Low-skill         271
Moderate-skill    228
Name: count, dtype: int64
✓ figures/fig2_elbow_silhouette.png and figures/fig2_elbow_silhouette.pdf
✓ figures/fig3_clusters_new.png and figures/fig3_clusters_new.pdf


In [14]:
# ============================================================
# SECTION 3e — K-SENSITIVITY ANALYSIS (k=2..5)
# ============================================================

print('\n=== Section 3e: k-sensitivity ===')
sens_rows = []
for k in range(2, 6):
    km_k  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl_k = km_k.fit_predict(X_sc)
    df_full[f'Cluster_k{k}'] = df_cl.index.map(
        lambda i: lbl_k[df_cl.index.get_loc(i)] if i in df_cl.index else np.nan)
    sil = silhouette_score(X_sc, lbl_k)
    ch  = calinski_harabasz_score(X_sc, lbl_k)
    db  = davies_bouldin_score(X_sc, lbl_k)
    n_each = pd.Series(lbl_k).value_counts().sort_index().tolist()
    sens_rows.append({'k': k, 'Silhouette': sil, 'CH': ch, 'DB': db,
                      'Sizes': n_each})
    print(f"k={k}: Sil={sil:.3f} CH={ch:.1f} DB={db:.3f} sizes={n_each}")

# ── Figure 4: k-sensitivity (publication layout; no overlapping layers) ──
# A vertical layout remains readable after scaling in a single- or double-column PDF.
ks_s = [r['k'] for r in sens_rows]
metric_specs = [
    ('Silhouette', 'Silhouette score', '#1976D2', '{:.3f}'),
    ('CH',         'Calinski–Harabasz index', '#0277BD', '{:.1f}'),
    ('DB',         'Davies–Bouldin index', '#00796B', '{:.3f}'),
]

fig, axes = plt.subplots(3, 1, figsize=(10.5, 13.5), sharex=True)

for ax, (metric, label, col, value_fmt) in zip(axes, metric_specs):
    vals = np.asarray([r[metric] for r in sens_rows], dtype=float)
    bars = ax.bar(
        ks_s, vals, width=0.62,
        color=col, alpha=0.86,
        edgecolor='white', linewidth=1.0,
        zorder=3
    )

    # Selected solution marker is drawn behind annotations and bars.
    ax.axvline(3, color='#D32F2F', linestyle='--', linewidth=1.8,
               alpha=0.85, zorder=2, label='Selected k = 3')

    # Reserve explicit vertical space above bars so value labels never collide.
    vmin, vmax = float(vals.min()), float(vals.max())
    span = max(vmax - min(0.0, vmin), abs(vmax) * 0.12, 1e-9)
    upper = vmax + span * 0.30
    lower = min(0.0, vmin - span * 0.08)
    ax.set_ylim(lower, upper)

    label_offset = span * 0.055
    for bar, value in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + label_offset,
            value_fmt.format(value),
            ha='center', va='bottom',
            fontsize=13, fontweight='bold',
            clip_on=False, zorder=5
        )

    ax.set_ylabel(label, fontsize=14, labelpad=12)
    ax.tick_params(axis='both', labelsize=13, pad=6)
    ax.grid(axis='y', alpha=0.25, linewidth=0.8, zorder=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(loc='upper right', fontsize=11, frameon=False)

axes[-1].set_xlabel('Number of clusters (k)', fontsize=15, labelpad=10)
axes[-1].set_xticks(ks_s)
axes[-1].set_xticklabels([str(k) for k in ks_s], fontsize=13)

fig.suptitle(
    'Cluster Validity Across Candidate Numbers of Clusters',
    fontsize=20, fontweight='bold', y=0.982
)
fig.subplots_adjust(left=0.18, right=0.97, bottom=0.075,
                    top=0.925, hspace=0.38)

save_journal_figure(fig, 'fig4_k_sensitivity')
plt.close(fig)



=== Section 3e: k-sensitivity ===
k=2: Sil=0.199 CH=241.9 DB=1.807 sizes=[422, 411]
k=3: Sil=0.185 CH=202.7 DB=1.835 sizes=[334, 271, 228]
k=4: Sil=0.171 CH=167.1 DB=1.760 sizes=[146, 266, 309, 112]
k=5: Sil=0.158 CH=143.7 DB=1.910 sizes=[285, 142, 109, 129, 168]
✓ figures/fig4_k_sensitivity.png and figures/fig4_k_sensitivity.pdf


In [15]:
# ============================================================
# SECTION 3f — GMM CLUSTERING ROBUSTNESS
# ============================================================

print('\n=== Section 3f: GMM robustness ===')
gmm_rows = []
for k in range(2, 6):
    gmm   = GaussianMixture(n_components=k, covariance_type='full',
                            random_state=42, n_init=5)
    lbl_g = gmm.fit_predict(X_sc)
    sil   = silhouette_score(X_sc, lbl_g)
    ch    = calinski_harabasz_score(X_sc, lbl_g)
    db    = davies_bouldin_score(X_sc, lbl_g)
    bic   = gmm.bic(X_sc)
    aic   = gmm.aic(X_sc)
    n_g   = pd.Series(lbl_g).value_counts().sort_index().tolist()
    gmm_rows.append({'k': k, 'Silhouette': sil, 'CH': ch,
                     'DB': db, 'BIC': bic, 'AIC': aic, 'Sizes': n_g})
    print(f"GMM k={k}: Sil={sil:.3f} CH={ch:.1f} DB={db:.3f} "
          f"BIC={bic:.0f} AIC={aic:.0f} sizes={n_g}")

# ── GMM vs KMeans comparison at k=3 ──────────────────────────────────────
km3_lbl  = km3.labels_
gmm3     = GaussianMixture(n_components=3, covariance_type='full',
                           random_state=42, n_init=5)
gmm3_lbl = gmm3.fit_predict(X_sc)

from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(km3_lbl, gmm3_lbl)
print(f"\nAdjusted Rand Index (KMeans k=3 vs GMM k=3): {ari:.4f}")
print("(ARI=1.0 = perfect agreement, 0.0 = random)")

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), constrained_layout=True)
for ax, (lbl, title) in zip(axes, [
    (km3_lbl, f'K-Means (k=3)\nSilhouette={cv_df[cv_df["k"]==3]["silhouette"].values[0]:.3f}'),
    (gmm3_lbl, f'GMM (k=3)\nSilhouette={gmm_rows[1]["Silhouette"]:.3f}'),
]):
    order_l = pd.Series(lbl).value_counts().sort_values(ascending=False).index
    palette = ['#4472C4','#FF8C00','#2ECC71']
    for i, lbl_i in enumerate(order_l):
        mask = lbl == lbl_i
        ax.scatter(X_pca[mask,0], X_pca[mask,1], c=palette[i],
                   alpha=0.6, s=20, label=f'Cluster {i+1}')
    ax.set_title(title, fontsize=16, pad=12)
    ax.set_xlabel(f'PC1 ({ev[0]:.1%})', fontsize=13)
    ax.set_ylabel(f'PC2 ({ev[1]:.1%})', fontsize=13)
    ax.tick_params(axis='both', labelsize=12)
    ax.legend(fontsize=11); ax.grid(alpha=0.25)
plt.suptitle(f'Clustering Robustness: K-Means vs GMM (k=3)\n'
             f'Adjusted Rand Index = {ari:.3f}', fontsize=19, fontweight='bold', y=1.05)
plt.tight_layout()
save_journal_figure(fig, 'fig5_gmm_robustness')
plt.close()


=== Section 3f: GMM robustness ===
GMM k=2: Sil=0.167 CH=194.7 DB=2.000 BIC=30049 AIC=28768 sizes=[442, 391]
GMM k=3: Sil=0.078 CH=71.5 DB=3.910 BIC=22923 AIC=21000 sizes=[347, 339, 147]
GMM k=4: Sil=0.053 CH=39.6 DB=3.018 BIC=13074 AIC=10508 sizes=[181, 18, 491, 143]
GMM k=5: Sil=0.039 CH=33.8 DB=2.945 BIC=15582 AIC=12374 sizes=[338, 27, 145, 305, 18]

Adjusted Rand Index (KMeans k=3 vs GMM k=3): 0.2761
(ARI=1.0 = perfect agreement, 0.0 = random)
✓ figures/fig5_gmm_robustness.png and figures/fig5_gmm_robustness.pdf


In [16]:
# ============================================================
# SECTION 4 — PREDICTIVE MODELLING (GroupKFold — Option Y)
# ============================================================

print('\n=== Section 4: GroupKFold CV ===')

df_rf = df[RF_FEATS + [OUTCOME, FARMER_ID]].apply(
    pd.to_numeric, errors='coerce').dropna().copy()
X_rf   = df_rf[RF_FEATS].values
y_rf   = df_rf[OUTCOME].values
groups = df_rf[FARMER_ID].values

print(f"RF dataset: {len(df_rf)} rows, {df_rf[FARMER_ID].nunique()} farmers")

gkfold = GroupKFold(n_splits=5)

print(f"\n{'Model':<28} {'R²':>8} {'±SD':>7} {'RMSE':>8} {'MAE':>8}")
print('─'*57)
model_results = {}
for name, model in [
    ('Linear Regression', LinearRegression()),
    ('Ridge Regression',  Ridge(alpha=1.0)),
    ('Random Forest (GroupKFold)',
     RandomForestRegressor(**RF_PARAMS)),
]:
    r2   = cross_val_score(model, X_rf, y_rf, cv=gkfold,
                           groups=groups, scoring='r2')
    rmse = -cross_val_score(model, X_rf, y_rf, cv=gkfold, groups=groups,
                            scoring='neg_root_mean_squared_error')
    mae  = -cross_val_score(model, X_rf, y_rf, cv=gkfold, groups=groups,
                            scoring='neg_mean_absolute_error')
    model_results[name] = dict(R2=r2.mean(), SD=r2.std(),
                               RMSE=rmse.mean(), MAE=mae.mean())
    marker = ' ←' if 'Forest' in name else ''
    print(f"{name:<28} {r2.mean():>8.3f} {r2.std():>7.3f} "
          f"{rmse.mean():>8.3f} {mae.mean():>8.3f}{marker}")

# Train final RF on full data for SHAP
rf_final = RandomForestRegressor(**RF_PARAMS)
rf_final.fit(df_rf[RF_FEATS], y_rf)


=== Section 4: GroupKFold CV ===
RF dataset: 832 rows, 417 farmers

Model                              R²     ±SD     RMSE      MAE
─────────────────────────────────────────────────────────
Linear Regression               0.255   0.026    0.599    0.467
Ridge Regression                0.255   0.026    0.599    0.467
Random Forest (GroupKFold)      0.284   0.018    0.587    0.454 ←


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0

In [17]:
# ============================================================
# SECTION 4b — HYPERPARAMETER GRID SEARCH + ROBUSTNESS CHECK
# ============================================================

print('\n=== Section 4b: Grid search ===')

from sklearn.model_selection import GroupKFold
from itertools import product

param_grid = {
    'n_estimators': [300, 500],
    'max_depth': [5, 8, None],
    'min_samples_leaf': [5, 10, 20],
    'max_features': ['sqrt', 0.5],
}

gkfold4b = GroupKFold(n_splits=5)
grid_results = []
for n_est, max_d, min_leaf, max_feat in product(
        param_grid['n_estimators'], param_grid['max_depth'],
        param_grid['min_samples_leaf'], param_grid['max_features']):
    rf_grid = RandomForestRegressor(
        n_estimators=n_est, max_depth=max_d,
        min_samples_leaf=min_leaf, max_features=max_feat,
        random_state=42, n_jobs=-1)
    r2_grid = cross_val_score(rf_grid, X_rf, y_rf, cv=gkfold4b,
                              groups=groups, scoring='r2')
    grid_results.append({
        'n_estimators': n_est, 'max_depth': max_d,
        'min_samples_leaf': min_leaf, 'max_features': max_feat,
        'R2_mean': r2_grid.mean(), 'R2_std': r2_grid.std()
    })

grid_df = pd.DataFrame(grid_results).sort_values('R2_mean', ascending=False)
print(f"Grid search: {len(grid_df)} combinations tested\n")
print(grid_df.head(10).to_string(index=False))

# Selected (reported) settings: n_estimators=500, min_samples_leaf=10,
# max_features='sqrt', max_depth=None — see Table: Random Forest
# hyperparameter optimization in the manuscript.
selected = grid_df[
    (grid_df.n_estimators == 500) &
    (grid_df.min_samples_leaf == 10) &
    (grid_df.max_features == 'sqrt') &
    (grid_df.max_depth.isna())
]
print(f"\nReported settings (n_estimators=500, min_samples_leaf=10, "
      f"max_features='sqrt', max_depth=None):")
print(selected.to_string(index=False))

# ── Fixed-hyperparameter robustness check ────────────────────────────────
# Verifies that the reported R^2 is not an artefact of hyperparameter
# search noise: refit the exact selected configuration once more and
# report its mean +/- SD, matching the figure quoted in the manuscript
# (R^2 = 0.283 +/- 0.036).
print('\n--- Fixed-hyperparameter robustness check ---')
rf_robust = RandomForestRegressor(**RF_PARAMS)
r2_robust = cross_val_score(rf_robust, X_rf, y_rf, cv=gkfold4b,
                            groups=groups, scoring='r2')
print(f"Fixed RF_PARAMS -> R2 = {r2_robust.mean():.3f} +/- {r2_robust.std():.3f}")
print("(Matches Table: RF (ours) R2 = 0.283 (0.036) reported in the manuscript)")



=== Section 4b: Grid search ===
Grid search: 36 combinations tested

 n_estimators  max_depth  min_samples_leaf max_features  R2_mean   R2_std
          500        8.0                 5          0.5 0.299976 0.032010
          300        8.0                 5          0.5 0.299424 0.034152
          300        NaN                 5          0.5 0.299080 0.033794
          500        NaN                 5          0.5 0.297072 0.032778
          300        NaN                10          0.5 0.294623 0.025012
          500        NaN                10          0.5 0.294159 0.025422
          300        8.0                10          0.5 0.294003 0.024920
          500        8.0                10          0.5 0.293423 0.025763
          500        NaN                 5         sqrt 0.292465 0.023876
          300        5.0                 5          0.5 0.291952 0.029860

Reported settings (n_estimators=500, min_samples_leaf=10, max_features='sqrt', max_depth=None):
 n_estimators  max_

In [18]:
# ============================================================
# SECTION 5 — SHAP ANALYSIS
# ============================================================

print('\n=== Section 5: SHAP ===')

df_shap = df[SHAP_FEATS + [OUTCOME, FARMER_ID]].apply(
    pd.to_numeric, errors='coerce').dropna().copy()
X_shap  = df_shap[SHAP_FEATS].rename(columns=SHAP_RENAME)
y_shap  = df_shap[OUTCOME].values

rf_shap = RandomForestRegressor(**RF_PARAMS)
rf_shap.fit(X_shap, y_shap)

explainer = shap.TreeExplainer(rf_shap)
sv        = explainer.shap_values(X_shap)
sv_rf     = sv        # alias for robustness check
exp_rf    = explainer

mean_shap = pd.Series(np.abs(sv).mean(0), index=X_shap.columns)

# Global bar
mean_shap_sorted = mean_shap.sort_values()
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(mean_shap_sorted.index, mean_shap_sorted.values,
        color='#2980b9', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Mean absolute SHAP value', fontsize=14)
ax.set_title('SHAP Feature Importance for Skill Change', fontsize=18, pad=12)
ax.tick_params(axis='both', labelsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
save_journal_figure(fig, 'fig6_importance_new')
plt.close()

# Beeswarm
fig = plt.figure(figsize=(11, 8))
shap.summary_plot(sv, X_shap, plot_type='dot',
                  max_display=10, show=False)
plt.title('SHAP Beeswarm: Determinants of Skill Change', fontsize=18, fontweight='bold', pad=14)
plt.gca().tick_params(axis='both', labelsize=12)
plt.gca().set_xlabel(plt.gca().get_xlabel(), fontsize=14)
plt.tight_layout()
save_journal_figure(fig, 'fig8_shap_beeswarm')
plt.close()

# Attach cluster labels
df_shap['Cluster'] = df_full.loc[
    df_shap.index.intersection(df_full.index), 'Cluster']
df_shap = df_shap.dropna(subset=['Cluster'])
df_shap['Cluster'] = df_shap['Cluster'].astype(int)

# Cluster-level beeswarm — PeerJ-readable vertical layout
# Each cluster is placed on its own row so the figure remains legible when
# scaled to the width used in main.tex. The analytical results are unchanged.
panel_titles = [
    f'Cluster 1: Low-skill (n={(df_shap["Cluster"]==0).sum()})',
    f'Cluster 2: Moderate-skill (n={(df_shap["Cluster"]==1).sum()})',
    f'Cluster 3: High-skill (n={(df_shap["Cluster"]==2).sum()})',
]

fig, axes = plt.subplots(
    nrows=3, ncols=1, figsize=(11.5, 20.0),
    constrained_layout=False
)

for ci, ax in enumerate(axes):
    idx = df_shap[df_shap['Cluster'] == ci].index
    sv_sub = explainer.shap_values(X_shap.loc[idx])

    top5 = (
        pd.Series(np.abs(sv_sub).mean(0), index=X_shap.columns)
        .sort_values(ascending=False)
        .head(4)
        .index.tolist()
    )
    col_idx = [list(X_shap.columns).index(f) for f in top5]

    plt.sca(ax)
    shap.summary_plot(
        sv_sub[:, col_idx],
        X_shap.loc[idx, top5].values,
        feature_names=top5,
        plot_type='dot',
        show=False,
        plot_size=None,
        max_display=4,
        color_bar=True
    )

    ax.set_title(panel_titles[ci], fontsize=20, fontweight='bold', pad=12)
    ax.set_xlabel('SHAP value (impact on model output)', fontsize=16, labelpad=8)
    ax.set_ylabel('')
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=15, pad=6)

    for label in ax.get_yticklabels():
        label.set_fontsize(15)
        label.set_fontweight('semibold')

    ax.grid(axis='x', alpha=0.20, linewidth=0.8)

# Enlarge the three color-bar axes created by SHAP.
for cax in fig.axes:
    if cax not in list(axes):
        cax.tick_params(labelsize=13)
        cax.set_ylabel(cax.get_ylabel(), fontsize=14, labelpad=8)

fig.suptitle(
    'Cluster-Stratified SHAP: Segment-Specific Drivers',
    fontsize=23, fontweight='bold', y=0.995
)
fig.subplots_adjust(left=0.24, right=0.92, top=0.955, bottom=0.055, hspace=0.52)
save_journal_figure(fig, 'fig9_cluster_shap_new')
plt.close()

# ── 5f. SHAP Robustness: RF vs XGBoost ───────────────────────────────────
print('\n--- 5f: SHAP Robustness ---')
xgb = XGBRegressor(n_estimators=500, learning_rate=0.05,
                   max_depth=6, subsample=0.8, colsample_bytree=0.8,
                   random_state=42, n_jobs=-1, verbosity=0)
xgb.fit(X_shap, y_shap)
exp_xgb = shap.TreeExplainer(xgb)
sv_xgb  = exp_xgb.shap_values(X_shap)

imp_rf  = pd.Series(np.abs(sv_rf ).mean(0), index=X_shap.columns, name='RF')
imp_xgb = pd.Series(np.abs(sv_xgb).mean(0), index=X_shap.columns, name='XGBoost')
rho, pval = spearmanr(imp_rf.rank(ascending=False),
                      imp_xgb.rank(ascending=False))
top4_rf  = imp_rf.sort_values(ascending=False).head(4).index.tolist()
top4_xgb = imp_xgb.sort_values(ascending=False).head(4).index.tolist()
overlap  = set(top4_rf) & set(top4_xgb)
print(f"Spearman ρ = {rho:.4f}  (p={pval:.4e})")
print(f"Top-4 overlap: {sorted(overlap)}  ({len(overlap)}/4)")

compare = pd.concat([imp_rf, imp_xgb], axis=1).sort_values('RF', ascending=True)
fig, ax = plt.subplots(figsize=(10, 7))
y_pos   = np.arange(len(compare)); h = 0.36
ax.barh(y_pos+h/2, compare['RF'],      height=h,
        color='#1565C0', alpha=0.85, label='Random Forest')
ax.barh(y_pos-h/2, compare['XGBoost'], height=h,
        color='#00796B', alpha=0.85, label='XGBoost')
ax.set_yticks(y_pos); ax.set_yticklabels(compare.index, fontsize=12)
ax.set_xlabel('Mean absolute SHAP value', fontsize=14)
ax.set_title(f'SHAP Attribution Robustness: RF vs XGBoost\n'
             f'Spearman ρ = {rho:.3f}  |  Top-4 overlap = {len(overlap)}/4',
             fontsize=17, fontweight='bold', pad=12)
ax.legend(fontsize=12, loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
save_journal_figure(fig, 'fig7_shap_robustness')
plt.close()


=== Section 5: SHAP ===
✓ figures/fig6_importance_new.png and figures/fig6_importance_new.pdf
✓ figures/fig8_shap_beeswarm.png and figures/fig8_shap_beeswarm.pdf
✓ figures/fig9_cluster_shap_new.png and figures/fig9_cluster_shap_new.pdf

--- 5f: SHAP Robustness ---
Spearman ρ = 0.7407  (p=2.4452e-03)
Top-4 overlap: ['All_Skill', 'Avg_ProdManage']  (2/4)
✓ figures/fig7_shap_robustness.png and figures/fig7_shap_robustness.pdf


In [19]:
# ── 5g. SHAP attribution mass for top-4 features (per cluster) ──────────────
print('\n--- 5g: SHAP Attribution Mass (Top-4 features per cluster) ---')

# คำนวณ attribution mass สำหรับแต่ละ cluster
attribution_mass = []
for ci in range(3):
    idx = df_shap[df_shap['Cluster'] == ci].index
    if len(idx) == 0:
        continue
    sv_sub = explainer.shap_values(X_shap.loc[idx])
    mean_abs_shap = np.abs(sv_sub).mean(0)
    total_mass = mean_abs_shap.sum()
    
    # top-4 features ตาม paper
    top4_cluster = ['All_Skill', 'Avg_ProdManage', 'Education', 'Avg_Mkting']
    # หา index ของ features เหล่านี้ใน X_shap.columns
    top4_idx = [list(X_shap.columns).index(f) for f in top4_cluster if f in X_shap.columns]
    top4_mass = mean_abs_shap[top4_idx].sum()
    pct = (top4_mass / total_mass) * 100
    
    cluster_name = CLUSTER_NAMES[ci]
    print(f'{cluster_name:>15}: top-4 features capture {pct:.1f}% of SHAP attribution mass')
    attribution_mass.append({'cluster': cluster_name, 'percentage': pct})

print(f'\nAverage across clusters: {np.mean([m["percentage"] for m in attribution_mass]):.1f}%')


--- 5g: SHAP Attribution Mass (Top-4 features per cluster) ---
      Low-skill: top-4 features capture 62.4% of SHAP attribution mass
 Moderate-skill: top-4 features capture 60.0% of SHAP attribution mass
     High-skill: top-4 features capture 60.6% of SHAP attribution mass

Average across clusters: 61.0%


In [20]:
import shutil
# ============================================================
# SECTION 6 — SURROGATE DECISION TREE
# ============================================================

print('\n=== Section 6: Surrogate Decision Tree ===')

# SHAP-selected top-5 features
TOP4_FEATS = ['All_Skill', 'Avg_ProdManage', 'edu', 'Avg_Mkting']
top_feats = TOP4_FEATS  # override with fixed top-4 for consistency
# top_feats = mean_shap.sort_values(ascending=False).head(5).index.tolist()
feat_orig = [k for k, v in SHAP_RENAME.items() if v in top_feats]
print(f"Selected SHAP features requested: {top_feats}")

# ── 6a. In-sample (baseline — existing result) ────────────────────────────
X_tree_all = df_shap[feat_orig].rename(columns=SHAP_RENAME)
y_tree_all = (df_shap['Cluster'] == 2).astype(int)
farmer_all = df_shap[FARMER_ID].values

dt_full = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_full.fit(X_tree_all, y_tree_all)
# Always use the exact columns seen by the fitted tree. Some requested features
# may be absent after mapping/filtering, so this prevents name-count mismatch.
tree_feature_names = X_tree_all.columns.astype(str).tolist()
assert len(tree_feature_names) == dt_full.n_features_in_, (
    f'Feature-name mismatch: {len(tree_feature_names)} names for '     f'{dt_full.n_features_in_} fitted features'
)
print(f"Features used by surrogate tree ({len(tree_feature_names)}): {tree_feature_names}")
acc_insample = accuracy_score(y_tree_all, dt_full.predict(X_tree_all))
print(f"\n6a. In-sample accuracy: {acc_insample:.3f}  (n={len(X_tree_all)})")

# ── Figure 10: original depth-3 tree with PeerJ-readable typography ───────
# Use the SAME fitted depth-3 surrogate as the analysis. Only typography,
# label density, color tone, and spacing are changed; model rules remain intact.
figure_feature_names = X_tree_all.columns.astype(str).tolist()
assert len(figure_feature_names) == dt_full.n_features_in_

FIG10_LABELS = {
    'All_Skill': 'Overall\nskill',
    'Avg_ProdManage': 'Prod.\nmgmt.',
    'Avg_Mkting': 'Marketing',
    'edu': 'Education',
}
figure_feature_labels = [FIG10_LABELS.get(x, x) for x in figure_feature_names]

children_left = dt_full.tree_.children_left
children_right = dt_full.tree_.children_right
leaf_ids = np.where(children_left == -1)[0].tolist()
internal_ids = [i for i in range(dt_full.tree_.node_count) if i not in leaf_ids]

def _lighten_hex(hex_color, mix=0.58):
    """Blend a hex color with white to create softer node fills."""
    raw = hex_color.lstrip('#')
    alpha = ''
    if len(raw) == 8:
        raw, alpha = raw[:6], raw[6:]
    if len(raw) != 6:
        return hex_color
    r = int(raw[0:2], 16)
    g = int(raw[2:4], 16)
    b = int(raw[4:6], 16)
    r = int(round(r + (255 - r) * mix))
    g = int(round(g + (255 - g) * mix))
    b = int(round(b + (255 - b) * mix))
    return f'#{r:02x}{g:02x}{b:02x}{alpha}'

def _soften_dot_fillcolors(dot_text, mix=0.58):
    import re
    return re.sub(
        r'(fillcolor=")(#(?:[0-9A-Fa-f]{6}|[0-9A-Fa-f]{8}))(\")',
        lambda m: f'{m.group(1)}{_lighten_hex(m.group(2), mix=mix)}{m.group(3)}',
        dot_text,
    )

def _compact_dot_labels(dot_text):
    """Shorten repeated node-label terms to reduce box width cleanly."""
    replacements = {
        'samples = ': 'n = ',
        'value = [': 'dist = [',
        'class = ': 'class: ',
        'Low/Moderate': 'Low/Mod.',
        'Production management': 'Prod. mgmt.',
        'Overall skill': 'Overall skill',
    }
    for old, new in replacements.items():
        dot_text = dot_text.replace(old, new)
    return dot_text

def _remove_internal_dist_from_dot(dot_text, internal_node_ids):
    """Keep dist only on leaf nodes by removing it from internal-node labels."""
    for node_id in internal_node_ids:
        pattern = rf'({node_id} \[label=<.*?)(?:<br/>dist = \[[^\]]+\])(<br/>class: .*?\], fillcolor=)'
        dot_text = re.sub(pattern, r'\1\2', dot_text, flags=re.DOTALL)
        pattern2 = rf'({node_id} \[label=<.*?)(?:<br/>value = \[[^\]]+\])(<br/>class = .*?\], fillcolor=)'
        dot_text = re.sub(pattern2, r'\1\2', dot_text, flags=re.DOTALL)
    return dot_text

dot_data = export_graphviz(
    dt_full,
    out_file=None,
    feature_names=figure_feature_labels,
    class_names=['Low/Mod.', 'High'],
    filled=True,
    rounded=True,
    special_characters=True,
    impurity=False,
    proportion=True,
    precision=2,
    label='root',
)

# More horizontal space plus larger leaf boxes makes the terminal nodes easier
# to read once the figure is scaled to journal width.
graph_header = (
    "digraph Tree {\n"
    "graph [rankdir=TB, ranksep=0.49, nodesep=0.52, margin=0.02, pad=0.05, "
    "ordering=out, splines=polyline, outputorder=edgesfirst, "
    "bgcolor=\"white\", dpi=600, concentrate=false, "
    f"label=\"Surrogate Decision Tree (depth = 3, accuracy = {acc_insample:.3f})\", "
    "labelloc=t, fontsize=25, fontname=\"DejaVu Sans Bold\", "
    "size=\"14.8,10.4!\", ratio=compress];\n"
    "node [fontname=\"DejaVu Sans\", fontsize=14.8, "
    "margin=\"0.16,0.10\", penwidth=1.05, style=\"rounded,filled\", "
    "width=1.56, height=0.68, fixedsize=false];\n"
    "edge [fontname=\"DejaVu Sans\", fontsize=12, penwidth=1.05, "
    "arrowsize=0.52, minlen=0.56, labeldistance=1.05, labelangle=24];"
)
dot_data = dot_data.replace('digraph Tree {', graph_header)
dot_data = _compact_dot_labels(dot_data)
dot_data = _remove_internal_dist_from_dot(dot_data, internal_ids)
dot_data = _soften_dot_fillcolors(dot_data, mix=0.58)

# Compute node depths so the deepest leaf row can be adjusted separately.
node_depth = np.zeros(dt_full.tree_.node_count, dtype=int)
stack = [(0, 0)]
while stack:
    node_id, depth = stack.pop()
    node_depth[node_id] = depth
    left = children_left[node_id]
    right = children_right[node_id]
    if left != -1:
        stack.append((left, depth + 1))
    if right != -1:
        stack.append((right, depth + 1))
max_leaf_depth = int(node_depth[leaf_ids].max()) if leaf_ids else 0
deepest_leaf_ids = [nid for nid in leaf_ids if node_depth[nid] == max_leaf_depth]
other_leaf_ids = [nid for nid in leaf_ids if node_depth[nid] != max_leaf_depth]

# Internal nodes can be more compact now that dist is removed.
internal_overrides = '\n'.join(
    f'{node_id} [fontsize=14.0, margin="0.15,0.10", width=1.46, height=0.66];'
    for node_id in internal_ids
)
# Leaves are deliberately larger; the deepest leaves are slightly narrower so
# they can still fit without collisions.
leaf_overrides = '\n'.join(
    [f'{node_id} [fontsize=14.6, margin="0.21,0.15", width=1.80, height=0.86];'
     for node_id in other_leaf_ids] +
    [f'{node_id} [fontsize=14.0, margin="0.19,0.13", width=1.62, height=0.80];'
     for node_id in deepest_leaf_ids]
)
dot_data = dot_data.rsplit('}', 1)[0] + '\n' + internal_overrides + '\n' + leaf_overrides + '\n}'

dot_path = os.path.join(OUT, 'fig10_rules_new.dot')
with open(dot_path, 'w', encoding='utf-8') as f:
    f.write(dot_data)

try:
    subprocess.run(
        ['dot', '-Tpng', '-Gdpi=600', dot_path,
         '-o', os.path.join(OUT, 'fig10_rules_new.png')],
        check=True
    )
    subprocess.run(
        ['dot', '-Tpdf', dot_path,
         '-o', os.path.join(OUT, 'fig10_rules_new.pdf')],
        check=True
    )
    subprocess.run(
        ['dot', '-Tsvg', dot_path,
         '-o', os.path.join(OUT, 'fig10_rules_new.svg')],
        check=True
    )
    print('✓ Figure 10 exported using the original depth-3 model')
    print(
        f'  Internal nodes: no dist | leaf nodes keep dist | '
        f'large leaves: {len(leaf_ids)} | deepest leaves: {len(deepest_leaf_ids)}'
    )
except (FileNotFoundError, subprocess.CalledProcessError):
    # Matplotlib fallback when Graphviz is unavailable.
    fig, ax = plt.subplots(figsize=(25, 16))
    tree_artists = plot_tree(
        dt_full,
        feature_names=figure_feature_labels,
        class_names=['Low/Mod.', 'High'],
        filled=True,
        rounded=True,
        impurity=False,
        proportion=True,
        precision=2,
        fontsize=11,
        ax=ax
    )

    def _compact_artist_label(text, keep_dist=True):
        replacements = {
            'samples = ': 'n = ',
            'value = ': 'dist = ',
            'class = ': 'class: ',
            'Low/Moderate': 'Low/Mod.',
            'Production management': 'Prod. mgmt.',
            'Overall skill': 'Overall\nskill',
        }
        for old, new in replacements.items():
            text = text.replace(old, new)
        lines = text.split('\n')
        if not keep_dist:
            lines = [ln for ln in lines if not ln.strip().startswith('dist =')]
        return '\n'.join(lines)

    y_positions = []
    for node_id, artist in enumerate(tree_artists):
        is_leaf = node_id in leaf_ids
        try:
            artist.set_text(_compact_artist_label(artist.get_text(), keep_dist=is_leaf))
        except Exception:
            pass
        try:
            x, y = artist.get_position()
            y_positions.append((node_id, y))
        except Exception:
            pass

        try:
            artist.set_fontsize(14 if is_leaf else 12)
        except Exception:
            pass

        patch = artist.get_bbox_patch()
        if patch is not None:
            try:
                patch.set_boxstyle('round,pad=0.34' if is_leaf else 'round,pad=0.24')
            except (AttributeError, TypeError, ValueError):
                pass
            try:
                r, g, b, a = patch.get_facecolor()
                mix = 0.60
                patch.set_facecolor((r + (1-r)*mix, g + (1-g)*mix, b + (1-b)*mix, a))
            except Exception:
                pass

    # Adjust only the deepest leaf row: keep it larger than internal nodes but
    # slightly tighter than the upper leaf row to prevent any overlap.
    if y_positions:
        bottom_y = min(y for node_id, y in y_positions if node_id in leaf_ids)
        for node_id, artist in enumerate(tree_artists):
            if node_id in leaf_ids:
                try:
                    x, y = artist.get_position()
                except Exception:
                    continue
                if abs(y - bottom_y) < 1e-6:
                    try:
                        artist.set_fontsize(13)
                    except Exception:
                        pass
                    patch = artist.get_bbox_patch()
                    if patch is not None:
                        try:
                            patch.set_boxstyle('round,pad=0.28')
                        except (AttributeError, TypeError, ValueError):
                            pass

    ax.set_title(
        f'Surrogate Decision Tree (depth = 3, accuracy = {acc_insample:.3f})',
        fontsize=25,
        fontweight='bold',
        pad=18
    )
    save_journal_figure(fig, 'fig10_rules_new', dpi=600)
    plt.close()

# ── 6b. Farmer-level 80/20 train/test split (Option B) ───────────────────
print('\n6b. Farmer-level 80/20 held-out validation (Option B)')

# Clustering features available in SHAP dataset
SURR_CLUSTER_FEATS = ['age','edu','agri_long','irriga','loan',
                      'Avg_ProdManage','Avg_Tech','Avg_Mkting',
                      'Ave_MktRisk','Ave_FinRisk']

# Year=1 only — one row per farmer (true post-training outcome)
ALL_FEATS_Y1 = SHAP_FEATS + [OUTCOME, FARMER_ID]
df_y0 = df[df['Year']==0][ALL_FEATS_Y1].apply(
    pd.to_numeric, errors='coerce').dropna(
    subset=SHAP_FEATS+[OUTCOME]).drop_duplicates(
    subset=FARMER_ID).reset_index(drop=True)

unique_farmers = df_y0[FARMER_ID].unique()
np.random.seed(42); np.random.shuffle(unique_farmers)
n_train    = int(len(unique_farmers) * 0.80)
train_ids  = unique_farmers[:n_train]
test_ids   = unique_farmers[n_train:]

df_tr = df_y0[df_y0[FARMER_ID].isin(train_ids)].copy()
df_te = df_y0[df_y0[FARMER_ID].isin(test_ids )].copy()
print(f"Train: {len(df_tr)} farmers | Test: {len(df_te)} farmers")

# Cluster on TRAIN only (Option B)
scaler_tr    = StandardScaler()
X_tr_sc      = scaler_tr.fit_transform(df_tr[SURR_CLUSTER_FEATS])
km_tr        = KMeans(n_clusters=3, random_state=42, n_init=10)
df_tr = df_tr.copy()
df_tr['Cluster_raw'] = km_tr.fit_predict(X_tr_sc)
tr_order   = df_tr.groupby('Cluster_raw')['All_Skill'].mean().sort_values()
tr_lbl_map = {old: new for new, old in enumerate(tr_order.index)}
df_tr['Cluster'] = df_tr['Cluster_raw'].map(tr_lbl_map)
print(f"Train cluster sizes: {df_tr['Cluster'].value_counts().sort_index().to_dict()}")

# Assign test to nearest centroid
df_te = df_te.copy()
df_te['Cluster_raw'] = km_tr.predict(
    scaler_tr.transform(df_te[SURR_CLUSTER_FEATS]))
df_te['Cluster'] = df_te['Cluster_raw'].map(tr_lbl_map)
print(f"Test  cluster sizes: {df_te['Cluster'].value_counts().sort_index().to_dict()}")

# SHAP on TRAIN to select top-4 features
X_tr_shap = df_tr[SHAP_FEATS].rename(columns=SHAP_RENAME)
rf_tr     = RandomForestRegressor(**RF_PARAMS)
rf_tr.fit(X_tr_shap, df_tr[OUTCOME].values)
sv_tr     = shap.TreeExplainer(rf_tr).shap_values(X_tr_shap)
top5_tr   = (pd.Series(np.abs(sv_tr).mean(0), index=X_tr_shap.columns)
               .sort_values(ascending=False).head(4).index.tolist())
feat_orig_tr = [k for k, v in SHAP_RENAME.items() if v in top5_tr]
print(f"Train top-5 SHAP: {top5_tr}")

# Surrogate tree — TRAIN
X_surr_tr = df_tr[feat_orig_tr].rename(columns=SHAP_RENAME)
y_surr_tr = (df_tr['Cluster'] == 2).astype(int)
dt_val    = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_val.fit(X_surr_tr, y_surr_tr)
acc_train = accuracy_score(y_surr_tr, dt_val.predict(X_surr_tr))

# Surrogate tree — TEST
X_surr_te = df_te[feat_orig_tr].rename(columns=SHAP_RENAME)
y_surr_te = (df_te['Cluster'] == 2).astype(int)
acc_test  = accuracy_score(y_surr_te, dt_val.predict(X_surr_te))

print(f"\nTrain accuracy (80%):    {acc_train:.3f}  (n={len(df_tr)})")
print(f"Held-out test (20%):     {acc_test:.3f}  (n={len(df_te)})")
print(f"\n{classification_report(y_surr_te, dt_val.predict(X_surr_te), target_names=['Low/Mod','High'])}")

# ── 6c. 5-fold GroupKFold CV ──────────────────────────────────────────────
print('\n6c. 5-fold GroupKFold CV on surrogate tree')

X_cv_all = df_shap[feat_orig_tr].rename(columns=SHAP_RENAME)
y_cv_all = (df_shap['Cluster'] == 2).astype(int)
g_cv_all = df_shap[FARMER_ID].values

gkf_surr = GroupKFold(n_splits=5)
cv_accs  = []
for fold, (tri, tei) in enumerate(gkf_surr.split(
        X_cv_all, y_cv_all, groups=g_cv_all)):
    dt_f = DecisionTreeClassifier(max_depth=3, random_state=42)
    dt_f.fit(X_cv_all.iloc[tri], y_cv_all.iloc[tri])
    a = accuracy_score(y_cv_all.iloc[tei],
                       dt_f.predict(X_cv_all.iloc[tei]))
    cv_accs.append(a)
    print(f"  Fold {fold+1}: {a:.3f}")
print(f"\n5-fold GroupKFold CV: {np.mean(cv_accs):.3f} +/- {np.std(cv_accs):.3f}")

print("\n=== FINAL SUMMARY: Surrogate Tree Accuracy ===")
print(f"{'In-sample (full, n=817)':<40} {acc_insample:.3f}")
print(f"{'Train set (80% farmers)':<40} {acc_train:.3f}")
print(f"{'Held-out test (20% farmers)':<40} {acc_test:.3f}")
print(f"{'5-fold GroupKFold CV':<40} {np.mean(cv_accs):.3f} +/- {np.std(cv_accs):.3f}")


=== Section 6: Surrogate Decision Tree ===
Selected SHAP features requested: ['All_Skill', 'Avg_ProdManage', 'edu', 'Avg_Mkting']
Features used by surrogate tree (3): ['Avg_ProdManage', 'Avg_Mkting', 'All_Skill']

6a. In-sample accuracy: 0.933  (n=817)
✓ figures/fig10_rules_new.png and figures/fig10_rules_new.pdf

6b. Farmer-level 80/20 held-out validation (Option B)
Train: 332 farmers | Test: 84 farmers
Train cluster sizes: {0: 98, 1: 100, 2: 134}
Test  cluster sizes: {0: 34, 1: 30, 2: 20}
Train top-5 SHAP: ['All_Skill', 'Avg_Mkting', 'Avg_ProdManage', 'Avg_Tech']

Train accuracy (80%):    0.919  (n=332)
Held-out test (20%):     0.845  (n=84)

              precision    recall  f1-score   support

     Low/Mod       0.93      0.86      0.89        64
        High       0.64      0.80      0.71        20

    accuracy                           0.85        84
   macro avg       0.79      0.83      0.80        84
weighted avg       0.86      0.85      0.85        84


6c. 5-fold GroupKF

In [21]:
# ============================================================
# SECTION 7 — ABLATION STUDY
# ============================================================
# Compares the full SAR pipeline against two simplified baselines:
#   (1) Clustering-only  — threshold on All_Skill, no ML attribution
#   (2) RF-only           — global RF prediction threshold, no segmentation
#   (3) SAR (proposed)    — segmentation + SHAP + surrogate rules
# Reuses df_shap, X_shap, y_tree, feat_orig, dt_full, acc_insample
# already defined in Section 6.

print('\n=== Section 7: Ablation study ===')

# ── (1) Clustering-only: threshold on overall skill (All_Skill) ──────────
# Uses the mean All_Skill across the sample as a simple, parameter-free
# cut-off — no machine learning involved.
all_skill_vals = df_shap['All_Skill'].values
thresh_cluster = all_skill_vals.mean()
pred_cluster   = (all_skill_vals > thresh_cluster).astype(int)
acc_cluster    = accuracy_score(y_tree_all, pred_cluster)
print(f"[1] Clustering-only (All_Skill > {thresh_cluster:.3f}): "
      f"accuracy = {acc_cluster:.3f}")

# ── (2) RF-only: global prediction threshold, no segmentation ────────────
# Uses the Random Forest's continuous Ch_Skill prediction with a simple
# zero/median cut-off to flag 'high training potential' farmers, without
# any cluster-specific structure.
rf_pred_all     = rf_shap.predict(X_shap)
thresh_rf       = np.median(rf_pred_all)
pred_rf         = (rf_pred_all > thresh_rf).astype(int)
acc_rf_only     = accuracy_score(y_tree_all, pred_rf)
print(f"[2] RF-only (pred > {thresh_rf:.3f}): "
      f"accuracy = {acc_rf_only:.3f}")

# ── (3) SAR (proposed): segmentation + SHAP + surrogate rules ────────────
print(f"[3] SAR (proposed): accuracy = {acc_insample:.3f}")

# ── Summary table (matches Table: Ablation study in the manuscript) ─────
ablation_df = pd.DataFrame([
    {'Model': 'Clustering-only', 'Description': 'Baseline skill threshold',
     'Accuracy': acc_cluster},
    {'Model': 'RF (no seg.)',    'Description': 'Global prediction threshold',
     'Accuracy': acc_rf_only},
    {'Model': 'SAR (proposed)',  'Description': 'Segmentation + SHAP + rules',
     'Accuracy': acc_insample},
])
ablation_df['Accuracy (%)'] = (ablation_df['Accuracy'] * 100).round(1)
print("\n" + ablation_df[['Model', 'Description', 'Accuracy (%)']].to_string(index=False))



=== Section 7: Ablation study ===
[1] Clustering-only (All_Skill > 3.587): accuracy = 0.891
[2] RF-only (pred > 0.001): accuracy = 0.793
[3] SAR (proposed): accuracy = 0.933

          Model                 Description  Accuracy (%)
Clustering-only    Baseline skill threshold          89.1
   RF (no seg.) Global prediction threshold          79.3
 SAR (proposed) Segmentation + SHAP + rules          93.3


In [22]:
# ============================================================
# SUMMARY
# ============================================================
print('\n' + '='*55)
print('All figures saved to ./figures/')
print('='*55)
for fn, desc in [
    ('fig_elbow_silhouette.png',   '4-panel cluster validity'),
    ('fig2_clusters_new.png',      'PCA scatter k=3'),
    ('fig_k_sensitivity.png',      'k-sensitivity k=2..5  [NEW]'),
    ('fig_gmm_robustness.png',     'GMM vs K-Means  [NEW]'),
    ('fig3_importance_new.png',    'Global SHAP bar'),
    ('fig4_shap_new.png',          'Global SHAP beeswarm'),
    ('fig4b_cluster_shap_new.png', 'Cluster SHAP 3-panel'),
    ('fig_shap_robustness.png',    'RF vs XGBoost SHAP'),
    ('fig5_rules_new.png',         'Surrogate decision tree'),
]:
    print(f"  {fn:<38} {desc}")


All figures saved to ./figures/
  fig_elbow_silhouette.png               4-panel cluster validity
  fig2_clusters_new.png                  PCA scatter k=3
  fig_k_sensitivity.png                  k-sensitivity k=2..5  [NEW]
  fig_gmm_robustness.png                 GMM vs K-Means  [NEW]
  fig3_importance_new.png                Global SHAP bar
  fig4_shap_new.png                      Global SHAP beeswarm
  fig4b_cluster_shap_new.png             Cluster SHAP 3-panel
  fig_shap_robustness.png                RF vs XGBoost SHAP
  fig5_rules_new.png                     Surrogate decision tree
